# 01. Qwen3-14B Zero-shot 기준선

기존 독립 zero-shot Colab 코드를 출력 없이 정리한 변환본이다. 공식 훈련 파이프라인과
분리된 기준선이며, `RUN_ZERO_SHOT=True`일 때만 모델을 적재하고 생성한다.

- BF16 지원 L4/A100급 GPU 권장
- `MODEL_REVISION`은 40자리 commit SHA로 고정
- 신규 결과는 `artifacts/zero_shot/<RUN_TAG>/`에 기록


## 1. Colab 런타임 준비

권장 런타임은 A100급 GPU다. L4/T4에서도 4bit 추론은 시도할 수 있지만 속도와 메모리 여유가 작을 수 있다.

아래 셀은 Colab에서 필요한 패키지를 설치한다. 로컬에서 구조만 확인할 때는 실행하지 않아도 된다.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/Chika1472/Writing-validation.git"
# 이 노트북 변환 시점의 소스 commit. 새 코드를 사용할 때만 의도적으로 바꾼다.
REPO_REF = os.environ.get(
    "WRITING_VALIDATION_REPO_REF",
    "30c70cb628208e004515144cc999e81d794bbe95",
).strip()
COLAB_PROJECT_DIR = Path("/content/Writing-validation")
CLONE_IF_MISSING = True

# 긴 학습은 두 값을 True로 두어 세션이 끊겨도 동일 절대경로를 유지한다.
USE_GOOGLE_DRIVE = False
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/Writing-validation/artifacts")
DRIVE_HF_HOME = Path("/content/drive/MyDrive/Writing-validation/huggingface")


def run_process(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("$", shlex.join(command))
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
    )


if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
    if not (COLAB_PROJECT_DIR / ".git").exists():
        if not CLONE_IF_MISSING:
            raise FileNotFoundError(f"저장소가 없습니다: {COLAB_PROJECT_DIR}")
        run_process(["git", "clone", REPO_URL, COLAB_PROJECT_DIR])
        run_process(["git", "checkout", "--detach", REPO_REF], cwd=COLAB_PROJECT_DIR)
    PROJECT_ROOT = COLAB_PROJECT_DIR.resolve()
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT_ROOT = next(
        (candidate.resolve() for candidate in candidates if (candidate / "pyproject.toml").is_file()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("pyproject.toml이 있는 저장소 루트에서 실행하세요.")

os.chdir(PROJECT_ROOT)

if USE_GOOGLE_DRIVE:
    DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    artifact_link = PROJECT_ROOT / "artifacts"
    if artifact_link.exists() and not artifact_link.is_symlink():
        raise RuntimeError(
            "artifacts가 이미 실제 디렉터리입니다. 산출물을 이동/삭제하지 말고 "
            "새 RUN_NAMESPACE를 사용해 경로 계약을 유지하세요."
        )
    if not artifact_link.exists():
        artifact_link.symlink_to(DRIVE_ARTIFACT_DIR, target_is_directory=True)
    DRIVE_HF_HOME.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(DRIVE_HF_HOME)

# 변환 전 commit의 옛 설정 경로도 Colab Linux에서 안전하게 호환한다.
compatibility_links = [
    (PROJECT_ROOT / "데이터셋", PROJECT_ROOT / "dataset"),
    (PROJECT_ROOT / "qwen3_14b_zero_shot", PROJECT_ROOT / "result" / "qwen3_14b_zero_shot"),
]
if IN_COLAB:
    for alias, target in compatibility_links:
        if not alias.exists() and target.exists():
            alias.symlink_to(target, target_is_directory=True)


def run_cli(label: str, enabled: bool, *arguments: object, env=None):
    command = [sys.executable, *[str(argument) for argument in arguments]]
    print(f"\n[{label}]", "RUN" if enabled else "SKIP")
    print("$", shlex.join(command))
    if not enabled:
        return None
    return subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


def require_paths(*paths: Path) -> None:
    missing = [str(path) for path in paths if not Path(path).exists()]
    if missing:
        raise FileNotFoundError("필수 파일이 없습니다: " + ", ".join(missing))


def require_model_revision(value: str, *, enabled: bool) -> None:
    if enabled and re.fullmatch(r"[0-9a-fA-F]{40}", value or "") is None:
        raise ValueError("MODEL_REVISION에 40자리 Hugging Face commit SHA를 입력하세요.")


def require_bf16_gpu(*, enabled: bool, recommended_vram_gb: float = 22.0) -> None:
    if not enabled:
        print("GPU 단계가 꺼져 있어 BF16 검사를 건너뜁니다.")
        return
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU가 필요합니다. Colab 런타임을 GPU로 변경하세요.")
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("이 파이프라인은 BF16이 필요합니다. T4 대신 L4/A100급을 선택하세요.")
    properties = torch.cuda.get_device_properties(0)
    vram_gb = properties.total_memory / 1024**3
    print({"gpu": properties.name, "vram_gb": round(vram_gb, 1), "bf16": True})
    if vram_gb < recommended_vram_gb:
        print(f"경고: 권장 VRAM {recommended_vram_gb:.0f}GB보다 작아 OOM 가능성이 큽니다.")


def show_json(path: Path):
    path = Path(path)
    if not path.exists():
        print("없음:", path)
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(payload, ensure_ascii=False, indent=2)[:12000])
    return payload


commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if IN_COLAB and commit != REPO_REF:
    raise RuntimeError(
        f"기존 Colab checkout {commit}이 고정 REPO_REF {REPO_REF}와 다릅니다. "
        "새 런타임/새 PROJECT_DIR를 사용하거나 모든 노트북에서 같은 ref를 지정하세요."
    )
print({
    "in_colab": IN_COLAB,
    "project_root": str(PROJECT_ROOT),
    "git_commit": commit,
    "artifacts": str((PROJECT_ROOT / "artifacts").resolve()),
    "hf_home": os.environ.get("HF_HOME"),
})


In [ ]:
INSTALL_DEPENDENCIES = IN_COLAB
PROJECT_EXTRAS = "qwen,test"

if INSTALL_DEPENDENCIES:
    run_process(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f".[{PROJECT_EXTRAS}]"],
        cwd=PROJECT_ROOT,
    )
else:
    print("로컬 검증에서는 기존 환경을 사용합니다.")


In [ ]:
RUN_ZERO_SHOT = False
MODEL_REVISION = os.environ.get("MODEL_REVISION", "").strip()
ALLOW_MODEL_DOWNLOAD = False

require_model_revision(MODEL_REVISION, enabled=RUN_ZERO_SHOT)
require_bf16_gpu(enabled=RUN_ZERO_SHOT)


## 2. 설정

`PROJECT_ROOT`는 `데이터셋` 폴더를 포함하는 프로젝트 루트다. Colab Drive를 쓰는 경우 Drive를 마운트한 뒤 경로 후보에 자동으로 잡히는지 확인한다.

전체 점수를 보려면 `MAX_TRAIN_ROWS=None`, `MAX_VALIDATION_ROWS=None`을 유지한다. 먼저 동작 확인만 하려면 각각 5~20 정도로 줄인다.


In [ ]:
import math
import random
import time
import hashlib

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MODEL_ID = "Qwen/Qwen3-14B"
RUN_TAG_BASE = "qwen3_14b_zero_shot_colab_v1"
PROMPT_CONTRACT_VERSION = "zero_shot_strict_v2"
MAX_TRAIN_ROWS = None
MAX_VALIDATION_ROWS = None
MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 768
GENERATION_BATCH_SIZE = 1
DOMAINS = ["content", "organization", "expression"]

DATA_DIR = PROJECT_ROOT / "dataset"
TRAIN_PATH = DATA_DIR / "글쓰기채점능력평가2026_train.jsonl"
VALIDATION_PATH = DATA_DIR / "글쓰기채점능력평가2026_validation.jsonl"
OUTPUT_ROOT = PROJECT_ROOT / "artifacts" / "zero_shot"
require_paths(TRAIN_PATH, VALIDATION_PATH)
print({"train": str(TRAIN_PATH), "validation": str(VALIDATION_PATH), "output_root": str(OUTPUT_ROOT)})


## 3. 데이터 로드

데이터는 JSONL이며, 각 행은 `prompt`, `essay`, `score`를 포함한다. `score`에는 세 영역과 평균 점수가 들어 있다.


In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def to_dataframe(rows: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    for domain in DOMAINS:
        df[f"gold_{domain}"] = df["score"].apply(lambda x: float(x[domain]))
    df["gold_average"] = df["score"].apply(lambda x: float(x.get("average", np.mean([x[d] for d in DOMAINS]))))
    return df

train_df = to_dataframe(read_jsonl(TRAIN_PATH))
validation_df = to_dataframe(read_jsonl(VALIDATION_PATH))

print("train", train_df.shape)
print("validation", validation_df.shape)
display(train_df[["id", "prompt_num", "gold_content", "gold_organization", "gold_expression", "gold_average"]].head())


In [ ]:
def score_distribution(df: pd.DataFrame, name: str) -> pd.DataFrame:
    rows = []
    for col in ["gold_content", "gold_organization", "gold_expression", "gold_average"]:
        rows.append({
            "split": name,
            "target": col,
            "count": len(df),
            "mean": df[col].mean(),
            "std": df[col].std(),
            "min": df[col].min(),
            "max": df[col].max(),
        })
    return pd.DataFrame(rows)

display(pd.concat([
    score_distribution(train_df, "train"),
    score_distribution(validation_df, "validation"),
], ignore_index=True).round(4))


## 4. 모델 로드

이 기준선은 학습, calibration, 규칙 기반 fallback을 하지 않는다. 모델이 생성한 텍스트에서 JSON만 파싱하고, 파싱 실패는 실패로 기록한다.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = None
model = None
if RUN_ZERO_SHOT:
    compute_dtype = torch.bfloat16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        trust_remote_code=False,
        local_files_only=not ALLOW_MODEL_DOWNLOAD,
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=compute_dtype,
        trust_remote_code=False,
        local_files_only=not ALLOW_MODEL_DOWNLOAD,
    )
    model.eval()
    print("loaded", MODEL_ID, MODEL_REVISION)
else:
    print("RUN_ZERO_SHOT=False: 모델 적재를 건너뜁니다.")


## 5. 프롬프트와 파서

프롬프트는 제출 형식에 맞춰 JSON만 요구한다. 점수는 1~5 범위의 숫자로 받는다. 여기서는 점수 보정이나 fallback을 하지 않으므로, 구조가 깨지면 그대로 실패로 계산한다.


In [ ]:
SYSTEM_PROMPT = '''너는 한국어 논증적 글을 일관되게 직접 채점하는 평가자이다.
essay를 읽고 content, organization, expression 세 기준을 모두 평가하라.
반드시 JSON 객체만 출력하고, Markdown 코드블록이나 추가 설명은 출력하지 마라.
생각 과정은 출력하지 마라.
5점은 매우 우수한 글에만 부여하고, 일반적으로 잘 쓴 글은 3~4점으로 평가하라.
근거가 일반적이거나 구체성이 부족하면 content를 낮추고, 구조는 있으나 연결과 전개가 약하면 organization을 낮추며, 문법·어휘·문장 흐름 문제가 보이면 expression을 낮춰라.
관대하게 점수를 올리지 말고, 감점 사유를 먼저 확인한 뒤 종합 판단하라.
각 score는 1 이상 5 이하의 숫자여야 한다.
rationale은 실제 essay 내용에 근거해 한국어 1~2문장으로 작성하라.
'''.strip()


def build_user_prompt(row: pd.Series) -> str:
    return f'''[평가 기준]
content: 주제 이해, 입장 명확성, 주장과 근거의 타당성, 논증의 깊이
organization: 서론-본론-결론 구조, 문단 구성, 전개 순서, 연결성
expression: 문장 정확성, 어휘 선택, 문법, 표현의 자연스러움

[점수 앵커]
5: 결함이 거의 없고, 주장·근거·구성·표현이 모두 매우 우수함
4: 대체로 우수하지만 일부 근거의 구체성, 연결, 표현상 아쉬움이 있음
3: 기본 요건은 충족하지만 근거가 일반적이거나 전개·표현의 약점이 뚜렷함
2: 입장, 근거, 구성, 표현 중 여러 요소가 부족해 설득력이 낮음
1: 주제 이해, 논증 구조, 문장 표현이 전반적으로 매우 미흡함

[출력 형식]
{{
  "content": {{"score": 1, "rationale": "content 판단 근거"}},
  "organization": {{"score": 1, "rationale": "organization 판단 근거"}},
  "expression": {{"score": 1, "rationale": "expression 판단 근거"}}
}}

[prompt_text]
{row['prompt']}

[essay_text]
{row['essay']}
'''.strip()


def apply_chat_template_no_thinking(messages: list[dict]) -> str:
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def make_prompt(row: pd.Series) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row)},
    ]
    return apply_chat_template_no_thinking(messages)



RUN_CONTRACT = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "prompt_contract_version": PROMPT_CONTRACT_VERSION,
    "system_prompt_sha256": hashlib.sha256(SYSTEM_PROMPT.encode("utf-8")).hexdigest(),
    "seed": SEED,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "generation_batch_size": GENERATION_BATCH_SIZE,
    "do_sample": False,
}
RUN_SIGNATURE = hashlib.sha256(
    json.dumps(RUN_CONTRACT, ensure_ascii=False, sort_keys=True).encode("utf-8")
).hexdigest()
RUN_TAG = f"{RUN_TAG_BASE}_{RUN_SIGNATURE[:12]}"
OUTPUT_DIR = OUTPUT_ROOT / RUN_TAG
if RUN_ZERO_SHOT:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({"run_tag": RUN_TAG, "run_signature": RUN_SIGNATURE, "output": str(OUTPUT_DIR)})


In [ ]:
def repair_common_json_errors(text: str) -> str:
    first = text.find("{")
    last = text.rfind("}")
    candidate = text[first:last + 1] if first >= 0 and last > first else text

    # Common Qwen typo: rationale string closes with ")" instead of "}"
    candidate = re.sub(
        r'("rationale"\s*:\s*"[^"\\]*(?:\\.[^"\\]*)*")\s*\)',
        r"\1}",
        candidate,
        flags=re.S,
    )

    # Remove trailing commas before object/list close
    candidate = re.sub(r",\s*([}\]])", r"\1", candidate)
    return candidate


def extract_first_json_object(text: str):
    decoder = json.JSONDecoder()

    repaired = repair_common_json_errors(text)
    try:
        return decoder.raw_decode(repaired[repaired.find("{"):])[0]
    except Exception:
        pass

    # Fallback: scan every opening brace, but only accept full schema later.
    for i, ch in enumerate(repaired):
        if ch != "{":
            continue
        try:
            obj, _ = decoder.raw_decode(repaired[i:])
            if isinstance(obj, dict) and all(d in obj for d in DOMAINS):
                return obj
        except json.JSONDecodeError:
            continue

    raise ValueError("No valid JSON object found")


def extract_domain_by_regex(text: str, domain: str) -> dict:
    m = re.search(rf'"{domain}"\s*:\s*\{{', text)
    if not m:
        raise ValueError(f"missing object: {domain}")

    body = text[m.end():]
    next_domain = re.search(r',\s*"(content|organization|expression)"\s*:', body)
    if next_domain:
        body = body[:next_domain.start()]

    score_m = re.search(
        r'"score"\s*:\s*(?:"(-?(?:\d+(?:\.\d*)?|\.\d+))"|(-?(?:\d+(?:\.\d*)?|\.\d+)))(?=\s*[,}])',
        body,
    )
    rationale_m = re.search(r'"rationale"\s*:\s*"((?:\\.|[^"\\])*)"', body, flags=re.S)

    if not score_m:
        raise ValueError(f"missing score: {domain}")
    if not rationale_m:
        raise ValueError(f"missing rationale: {domain}")

    score_text = score_m.group(1) or score_m.group(2)
    rationale_raw = rationale_m.group(1)
    try:
        rationale = json.loads(f'"{rationale_raw}"')
    except Exception:
        rationale = rationale_raw.replace('\\"', '"').strip()

    return {
        "score": float(score_text),
        "rationale": rationale,
    }


def validate_prediction_object(obj: dict) -> dict:
    if not isinstance(obj, dict):
        raise ValueError("prediction is not a JSON object")

    clean = {}
    for domain in DOMAINS:
        if domain not in obj or not isinstance(obj[domain], dict):
            raise ValueError(f"missing object: {domain}")

        score = obj[domain].get("score")
        rationale = obj[domain].get("rationale")

        if isinstance(score, str):
            score_text = score.strip()
            if re.fullmatch(r"-?(?:\d+(?:\.\d*)?|\.\d+)", score_text) is None:
                raise ValueError(f"non-numeric score: {domain}")
            score = float(score_text)

        if isinstance(score, bool) or not isinstance(score, (int, float)):
            raise ValueError(f"non-numeric score: {domain}")

        score = float(score)
        if not (1.0 <= score <= 5.0):
            raise ValueError(f"score out of range: {domain}={score}")

        if not isinstance(rationale, str) or not rationale.strip():
            raise ValueError(f"empty rationale: {domain}")

        clean[domain] = {
            "score": score,
            "rationale": rationale.strip(),
        }

    return clean


def parse_model_output(text: str) -> dict:
    try:
        obj = extract_first_json_object(text)
        clean = validate_prediction_object(obj)
        return {"ok": True, "parsed": clean, "error": None}
    except Exception as first_exc:
        try:
            repaired_text = repair_common_json_errors(text)
            clean = {
                domain: extract_domain_by_regex(repaired_text, domain)
                for domain in DOMAINS
            }
            clean = validate_prediction_object(clean)
            return {"ok": True, "parsed": clean, "error": None}
        except Exception as second_exc:
            return {
                "ok": False,
                "parsed": None,
                "error": f"{first_exc}; regex fallback failed: {second_exc}",
            }


def _parser_contract_probe(score_literal: str) -> str:
    return (
        '{"content":{"score":' + score_literal + ',"rationale":"근거"},'
        '"organization":{"score":3,"rationale":"근거"},'
        '"expression":{"score":3,"rationale":"근거"}}'
    )


assert parse_model_output(_parser_contract_probe("3.5"))["ok"]
for invalid_score in ("true", "10", "50", '\"10\"', '\"50\"', '\"1abc\"'):
    assert not parse_model_output(_parser_contract_probe(invalid_score))["ok"]


## 6. 단일 샘플 추론 확인

첫 샘플에서 출력 형식이 안정적인지 확인한다. 이 셀이 실패하면 전체 루프를 돌리지 말고 프롬프트나 `MAX_NEW_TOKENS`를 먼저 조정한다.


In [ ]:
@torch.inference_mode()
def generate_batch(rows: list[pd.Series]) -> list[str]:
    if model is None or tokenizer is None:
        raise RuntimeError("먼저 RUN_ZERO_SHOT=True로 모델을 적재하세요.")
    prompts = [make_prompt(row) for row in rows]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    prompt_width = inputs["input_ids"].shape[-1]
    return [
        tokenizer.decode(row[prompt_width:], skip_special_tokens=True).strip()
        for row in output_ids
    ]


def generate_one(row: pd.Series) -> str:
    return generate_batch([row])[0]


if RUN_ZERO_SHOT:
    sample_output = generate_one(validation_df.iloc[0])
    print(sample_output)
    print(parse_model_output(sample_output))
else:
    print("샘플 생성 생략")


## 7. 전체/부분 실행 루프

각 레코드는 생성 직후 JSONL로 저장된다. Colab 세션이 끊기면 같은 셀을 다시 실행해 이미 저장된 `id`를 건너뛸 수 있다. GPU 여유가 있으면 `GENERATION_BATCH_SIZE`를 3 또는 4로 올려 한 번에 여러 글을 생성한다.


In [ ]:
def select_rows(df: pd.DataFrame, limit: int | None) -> pd.DataFrame:
    if limit is None:
        return df.copy()
    return df.head(limit).copy()


def prediction_path(split_name: str) -> Path:
    return OUTPUT_DIR / f"{RUN_TAG}_{split_name}.jsonl"


def load_prediction_records(path: Path) -> list[dict]:
    if not path.exists():
        return []
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def iter_row_batches(df: pd.DataFrame, batch_size: int):
    for start in range(0, len(df), batch_size):
        yield [row for _, row in df.iloc[start:start + batch_size].iterrows()]


def run_split(df: pd.DataFrame, split_name: str) -> list[dict]:
    path = prediction_path(split_name)
    existing = load_prediction_records(path)
    mismatched = [
        row.get("id") for row in existing
        if row.get("run_signature") != RUN_SIGNATURE
    ]
    if mismatched:
        raise RuntimeError(
            f"output에 다른 revision/생성 계약의 행이 섞여 있습니다: {mismatched[:5]}"
        )
    done_ids = {
        row["id"] for row in existing
        if row.get("run_signature") == RUN_SIGNATURE
    }
    rows = df[~df["id"].isin(done_ids)].copy()
    print({
        "split": split_name,
        "target_rows": len(df),
        "already_done": len(done_ids),
        "remaining": len(rows),
        "batch_size": GENERATION_BATCH_SIZE,
        "path": str(path),
    })

    with path.open("a", encoding="utf-8") as f:
        for batch_rows in tqdm(
            iter_row_batches(rows, GENERATION_BATCH_SIZE),
            total=math.ceil(len(rows) / GENERATION_BATCH_SIZE) if len(rows) else 0,
            desc=split_name,
        ):
            started = time.time()
            raw_outputs = generate_batch(batch_rows)
            elapsed = time.time() - started
            per_item_elapsed = elapsed / max(len(batch_rows), 1)

            for row, raw_output in zip(batch_rows, raw_outputs):
                parsed = parse_model_output(raw_output)
                record = {
                    "run_tag": RUN_TAG,
                    "run_signature": RUN_SIGNATURE,
                    "run_contract": RUN_CONTRACT,
                    "model_id": MODEL_ID,
                    "model_revision": MODEL_REVISION,
                    "split": split_name,
                    "id": row["id"],
                    "prompt_num": row.get("prompt_num"),
                    "gold_score": row["score"],
                    "raw_output": raw_output,
                    "parse_ok": parsed["ok"],
                    "parsed": parsed["parsed"],
                    "parse_error": parsed["error"],
                    "elapsed_sec": round(per_item_elapsed, 3),
                    "batch_size": len(batch_rows),
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
            f.flush()

    return load_prediction_records(path)

train_eval_df = select_rows(train_df, MAX_TRAIN_ROWS)
validation_eval_df = select_rows(validation_df, MAX_VALIDATION_ROWS)

print("train_eval", train_eval_df.shape)
print("validation_eval", validation_eval_df.shape)


In [ ]:
# validation을 먼저 실행한다. append-resume는 같은 RUN_TAG의 완료 ID를 건너뛴다.
validation_records = (
    run_split(validation_eval_df, "validation")
    if RUN_ZERO_SHOT
    else load_prediction_records(prediction_path("validation"))
)
print("validation records", len(validation_records))


In [ ]:
RUN_TRAIN_DRIFT_CHECK = False
train_records = (
    run_split(train_eval_df, "train")
    if RUN_ZERO_SHOT and RUN_TRAIN_DRIFT_CHECK
    else load_prediction_records(prediction_path("train"))
)
print("train records", len(train_records))


## 8. 평가 지표 계산

이 평가는 raw zero-shot 기준선이다. calibration, threshold 조정, fallback 점수 대입을 하지 않는다. 따라서 파싱 실패율도 점수와 함께 봐야 한다.


In [ ]:
def records_to_pred_df(records: list[dict]) -> pd.DataFrame:
    rows = []
    for r in records:
        row = {
            "id": r["id"],
            "parse_ok": bool(r.get("parse_ok")),
            "parse_error": r.get("parse_error"),
            "elapsed_sec": r.get("elapsed_sec"),
        }
        parsed = r.get("parsed") if r.get("parse_ok") else None
        if parsed:
            for domain in DOMAINS:
                row[f"pred_{domain}"] = float(parsed[domain]["score"])
            row["pred_average"] = float(np.mean([row[f"pred_{d}"] for d in DOMAINS]))
        rows.append(row)
    pred_df = pd.DataFrame(rows)
    if not pred_df.empty:
        pred_df = pred_df.drop_duplicates("id", keep="last")
    return pred_df


def rmse(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def summarize_metrics(gold_df: pd.DataFrame, records: list[dict], split_name: str):
    pred_df = records_to_pred_df(records)
    merged = gold_df.merge(pred_df, on="id", how="left")
    expected = len(gold_df)
    generated = int(merged["parse_ok"].notna().sum()) if "parse_ok" in merged else 0
    parsed = merged[merged["parse_ok"] == True].copy()

    metric_rows = []
    for target in DOMAINS + ["average"]:
        y_true_col = f"gold_{target}"
        y_pred_col = f"pred_{target}"
        if len(parsed) == 0 or y_pred_col not in parsed:
            metric_rows.append({"split": split_name, "target": target, "rmse": np.nan, "spearman": np.nan, "n": 0})
            continue
        y_true = parsed[y_true_col].astype(float)
        y_pred = parsed[y_pred_col].astype(float)
        corr = spearmanr(y_true, y_pred).correlation if y_pred.nunique() > 1 else np.nan
        metric_rows.append({
            "split": split_name,
            "target": target,
            "rmse": rmse(y_true, y_pred),
            "spearman": float(corr) if corr is not None else np.nan,
            "n": len(parsed),
        })

    coverage = {
        "split": split_name,
        "expected_rows": expected,
        "generated_rows": generated,
        "parsed_rows": len(parsed),
        "missing_rows": expected - generated,
        "parse_fail_rows": generated - len(parsed),
        "parse_success_rate": len(parsed) / generated if generated else 0.0,
        "mean_elapsed_sec": float(pd.to_numeric(merged.get("elapsed_sec"), errors="coerce").mean()),
    }
    return pd.DataFrame(metric_rows), coverage, merged

if validation_records:
    validation_metrics, validation_coverage, validation_joined = summarize_metrics(
        validation_eval_df, validation_records, "validation"
    )
    display(pd.DataFrame([validation_coverage]))
    display(validation_metrics.round(4))
else:
    validation_metrics = pd.DataFrame()
    validation_coverage = {"status": "not_run", "expected_rows": len(validation_eval_df)}
    validation_joined = validation_eval_df.copy()
    validation_joined["parse_ok"] = pd.NA
    validation_joined["parse_error"] = pd.NA
    print(validation_coverage)


In [ ]:
if validation_records:
    metrics_out = OUTPUT_DIR / f"{RUN_TAG}_metrics.csv"
    coverage_out = OUTPUT_DIR / f"{RUN_TAG}_coverage.json"
    validation_metrics.to_csv(metrics_out, index=False, encoding="utf-8-sig")
    coverage_out.write_text(
        json.dumps({"validation": validation_coverage}, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print(metrics_out, coverage_out)
else:
    print("저장할 실행 결과가 없습니다.")


## 9. 실패 사례와 샘플 확인

파싱 실패가 있으면 먼저 raw output을 보고 프롬프트를 조정한다. 점수 보정이나 fallback은 이 기준선 노트북에서는 적용하지 않는다.


In [ ]:
def show_failures(joined: pd.DataFrame, split_name: str, n: int = 5):
    fail = joined[(joined["parse_ok"].notna()) & (joined["parse_ok"] != True)].copy()
    print(split_name, "failures", len(fail))
    cols = ["id", "parse_error"]
    display(fail[cols].head(n))
    return fail.head(n)

validation_failures = show_failures(validation_joined, "validation")

In [ ]:
def show_prediction_samples(joined: pd.DataFrame, n: int = 5):
    cols = [
        "id", "prompt_num",
        "gold_content", "pred_content",
        "gold_organization", "pred_organization",
        "gold_expression", "pred_expression",
        "gold_average", "pred_average",
    ]
    available = [c for c in cols if c in joined.columns]
    display(joined[joined["parse_ok"] == True][available].head(n))

show_prediction_samples(validation_joined, n=10)


In [ ]:
# Parse failure analysis
fail_df = validation_joined[
    (validation_joined["parse_ok"].notna()) & (validation_joined["parse_ok"] != True)
].copy()

print("failures:", len(fail_df))
display(fail_df["parse_error"].value_counts().head(20))

# raw output 확인
fail_ids = set(fail_df["id"])
fail_records = [r for r in validation_records if r["id"] in fail_ids]

for r in fail_records[:10]:
    print("=" * 80)
    print("id:", r["id"])
    print("error:", r["parse_error"])
    print(r["raw_output"][:2000])

In [ ]:
parsed_df = validation_joined[validation_joined["parse_ok"] == True].copy()
if parsed_df.empty:
    print("표시할 예측이 없습니다.")
else:
    for domain in DOMAINS:
        print("\n", domain)
        display(pd.DataFrame({
            "gold": parsed_df[f"gold_{domain}"].round(2),
            "pred": parsed_df[f"pred_{domain}"].round(2),
        }).describe().round(3))
        display(parsed_df[f"pred_{domain}"].value_counts().sort_index())
